In [ ]:
# IMPORTS
from egomimic.rldb.utils import *
import torch
import numpy as np
from egomimic.utils.egomimicUtils import CameraTransforms, draw_actions
import torchvision.io as io
import os

In [ ]:
# Load dataset
root = "/coc/flash7/paphiwetsa3/datasets/eva_test_data2/proc2/lerobot_test"
repo_id = "rpuns/aria_laundry_rl2"

episodes = [0, 1]
dataset = RLDBDataset(
    repo_id=repo_id, root=root, local_files_only=True, episodes=episodes, mode="sample"
)

In [ ]:
# Get metadata
print(dataset.meta.info["features"])

image_key = "observations.images.front_img_1"
actions_key = "actions_cartesian"

In [ ]:
print(dataset.embodiment)

In [ ]:
# Make data_loader
data_loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

In [19]:
camera_transforms = CameraTransforms(intrinsics_key="base", extrinsics_key="x5Nov18_3")

In [ ]:
camera_transforms.extrinsics["left"]

In [21]:
def visualize_actions(ims, actions, extrinsics, intrinsics, arm="both"):
    for b in range(ims.shape[0]):
        if actions.shape[-1] == 7 or actions.shape[-1] == 14:
            ac_type = "joints"
        elif actions.shape[-1] == 3 or actions.shape[-1] == 6:
            ac_type = "xyz"
        else:
            raise ValueError(f"Unknown action type with shape {actions.shape}")

        ims[b] = draw_actions(ims[b], "xyz", "Purples", actions[b], extrinsics, intrinsics, arm=arm)

    return ims

In [ ]:
save_dir = "./visualization/"
os.makedirs(save_dir, exist_ok=True)

num_batches = 1

for i, data in enumerate(data_loader):
    if i > num_batches:
        break
    ims = (data[image_key].permute(0, 2, 3, 1).cpu().numpy() * 255.0).astype(np.uint8)
    actions = data[actions_key].cpu().numpy()
    # print(actions_key)
    print(actions[:10, :])

    ims_viz = visualize_actions(ims, actions[:, :3], camera_transforms.extrinsics, camera_transforms.intrinsics)

    for j, im in enumerate(ims_viz):
        img_tensor = torch.from_numpy(im).permute(2, 0, 1)
        save_path = os.path.join(save_dir, f"image_{i}_{j}.png")
        io.write_png(img_tensor, save_path)

    print(f"Saved batch {i} images to {save_dir}")
    break